In [2]:
import torch
from weaver.nn.model.ParticleTransformer import ParticleTransformer
from weaver.utils.logger import _logger

'''
Link to the full model implementation:
https://github.com/hqucms/weaver-core/blob/main/weaver/nn/model/ParticleTransformer.py
'''


class ParticleTransformerWrapper(torch.nn.Module):
    def __init__(self, **kwargs) -> None:
        super().__init__()
        self.mod = ParticleTransformer(**kwargs)

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'mod.cls_token', }

    def forward(self, points, features, lorentz_vectors, mask):
        return self.mod(features, v=lorentz_vectors, mask=mask)


def get_model(**kwargs):

    cfg = dict(
        input_dim=17,
        num_classes=10,
        # network configurations
        pair_input_dim=4,
        use_pre_activation_pair=False,
        embed_dims=[128, 512, 128],
        pair_embed_dims=[64, 64, 64],
        num_heads=8,
        num_layers=8,
        num_cls_layers=2,
        block_params=None,
        cls_block_params={'dropout': 0, 'attn_dropout': 0, 'activation_dropout': 0},
        fc_params=[],
        activation='gelu',
        # misc
        trim=True,
        for_inference=False,
    )
    
    cfg.update(**kwargs)
    _logger.info('Model config: %s' % str(cfg))

    model = ParticleTransformerWrapper(**cfg)

    return model


def get_loss(data_config, **kwargs):
    return torch.nn.CrossEntropyLoss()


In [3]:
jetclass_model = get_model()
jetclass_part_statedict = torch.load('../models/ParT_full.pt', map_location='cpu')
jetclass_model.load_state_dict(jetclass_part_statedict)

<All keys matched successfully>

In [4]:
import numpy as np
howmanyjets = 5

jc_full_pf_features = np.load('../jc_full_data/jc_full_pf_features.npy')[:howmanyjets]
jc_full_pf_vectors = np.load('../jc_full_data/jc_full_pf_vectors.npy')[:howmanyjets]
jc_full_pf_mask = np.load('../jc_full_data/jc_full_pf_mask.npy')[:howmanyjets]
jc_full_pf_points = np.load('../jc_full_data/jc_full_pf_points.npy')[:howmanyjets]
jc_full_labels = np.load('../jc_full_data/jc_full_labels.npy')[:howmanyjets]

torch.manual_seed(42)

In [53]:
with torch.no_grad():
    y = jetclass_model(torch.from_numpy(jc_full_pf_points),torch.from_numpy(jc_full_pf_features),
                       torch.from_numpy(jc_full_pf_vectors),torch.from_numpy(jc_full_pf_mask))

print(y)

tensor([[ 5.3178e-02,  1.6453e+00,  4.9876e-01,  9.8855e-01, -2.9566e+00,
         -5.5194e+00,  2.1823e+00, -1.2363e+00, -3.9830e+00, -4.6772e+00],
        [ 2.1878e+00,  1.4119e+00, -4.9683e-01, -3.6859e-01, -3.7595e+00,
         -3.8776e+00,  1.9586e+00, -6.8634e-01, -1.9808e+00, -2.6787e+00],
        [ 7.2272e-01,  1.7237e+00,  1.6108e-01,  1.3934e+00, -4.3801e-01,
          6.9049e-02, -2.8023e-02, -1.4422e+00, -2.0338e+00, -7.4914e-02],
        [ 3.8298e-03,  4.6216e+00, -1.0366e-01,  2.2356e+00, -3.5295e+00,
         -3.4756e+00,  1.2425e+00, -1.6797e+00, -3.7311e+00,  8.6596e-02],
        [ 2.2938e+00,  2.4368e+00, -6.9961e-01,  1.1346e+00, -1.5872e+00,
         -4.7457e+00,  4.7491e-01, -2.3319e+00,  1.4766e+00, -1.1240e+00]])


In [26]:
dummy_vectors = torch.zeros_like(torch.from_numpy(jc_full_pf_vectors))

In [54]:
with torch.no_grad():
    y = jetclass_model(torch.from_numpy(jc_full_pf_points),torch.from_numpy(jc_full_pf_features),
                       dummy_vectors,torch.from_numpy(jc_full_pf_mask))

print(y)

tensor([[ 1.8038,  0.8919, -0.2858,  1.5566, -3.2292, -3.0901,  1.9818, -1.7544,
         -5.9794, -0.5514],
        [ 2.8980,  2.1038, -0.7600,  0.5336, -3.3705, -5.5714,  1.7681, -2.1716,
         -2.0469, -1.1816],
        [ 0.3683,  3.1372,  0.4386,  1.8464, -1.8532, -0.6445,  0.6990, -2.7119,
         -2.9914,  0.5580],
        [ 2.1661,  5.0459,  0.8176,  5.6849, -4.6648, -6.4861,  0.6659, -5.5360,
         -4.0911, -1.9295],
        [ 3.1083,  2.6078, -1.0801,  1.4734, -2.0503, -4.4689,  0.5223, -2.9902,
          0.9152,  1.2224]])


In [5]:
for param in jetclass_model.mod.pair_embed.parameters():
    param.data.zero_()

In [6]:
# print parameters in the pair_embed module to verify they are zeroed out
for name, module in jetclass_model.named_modules():
    if name == 'mod.pair_embed':
        for param_name, param in module.named_parameters():
            print(f'Parameter name: {param_name}, values: {param.data}')

Parameter name: embed.0.weight, values: tensor([0., 0., 0., 0.])
Parameter name: embed.0.bias, values: tensor([0., 0., 0., 0.])
Parameter name: embed.1.weight, values: tensor([[[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0

In [7]:
torch.save(jetclass_model.state_dict(),'../models/JetClass_ParT_zeroed_interaction.pt')